# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO DIRIGIDO N.° 06

### DISEÑO Y ANÁLISIS DE UN EXPERIMENTO A/B

**Semana 6 · TECSUP 2026-II**  
**Docente:** Pilar Rocío Sayán Mejía  
**Carrera:** Big Data y Ciencia de Datos  
**Plataforma de trabajo:** Google Colab

ESTUDIANTE: Chuquiyuri Valdivia Edwar
### Pregunta orientadora

**¿Qué evidencia estadística y práctica necesitamos antes de implementar una variante B?**

> Este notebook acompaña el trabajo desarrollado durante la clase. No es un informe ni un proyecto para exponer. Las celdas alternan explicación breve, ejecución guiada, interpretación y retos de cierre.


# I. Elemento de la capacidad terminal de la semana

Diseña y analiza un experimento A/B identificando el factor, sus niveles, el control, el tratamiento, la unidad experimental y la variable respuesta. Formula las hipótesis estadísticas, establece la métrica primaria, los guardrails, el nivel de significación, la potencia y el efecto mínimo detectable; finalmente, integra la evidencia estadística con la relevancia práctica para sustentar una decisión.


# II. Seguridad y condiciones de trabajo

- Utilice únicamente Google Colab y las fuentes indicadas por la docente.
- No modifique ni fabrique resultados para forzar una conclusión.
- Ejecute las celdas en el orden propuesto y documente cualquier error o dato ausente.
- No comparta claves, credenciales ni información personal en el notebook.
- Mantenga la estación de trabajo ordenada y no instale software no autorizado.


# III. Fundamento teórico
Un A/B test compara un control A con una variante B mediante asignación aleatoria. La unidad experimental de este caso es el usuario, que conserva su variante durante toda la observación. La aleatorización favorece la comparabilidad en expectativa; no garantiza grupos idénticos.

La respuesta es binaria: compra dentro de los siete días posteriores a la asignación. Todos los usuarios completan esa ventana antes del análisis. La métrica primaria es la proporción de usuarios que compran. Se estudia la diferencia de dos proporciones, no ANOVA.

El MDE es el efecto utilizado para planificar la potencia. El umbral empresarial es la mejora que justificaría interés práctico. Aquí son distintos: MDE de 1,5 pp y umbral empresarial de 1,0 pp. La muestra se dimensiona para detectar el MDE frente a cero, no para garantizar evidencia de superar el umbral empresarial.

Se usa una aproximación normal de Wald para la diferencia. El contraste unilateral al 5 % y el límite inferior unilateral del 95 % utilizan el mismo error estándar. También se muestra un IC bilateral del 95 % como descripción de incertidumbre; no se exige que este último excluya cero para reproducir el contraste unilateral.

La recomendación académica requiere evidencia de superar el umbral empresarial y de mantener el guardrail dentro del margen predefinido. Son criterios más exigentes que obtener p < 0,05 frente a cero. Un resultado no concluyente no demuestra igualdad.


# IV. Normas y buenas prácticas empleadas

- Prerregistro lógico: definir hipótesis, métrica, MDE y regla de decisión antes de observar los resultados.
- Aleatorización: evitar que la asignación dependa de características de las unidades.
- Trazabilidad: conservar la fuente, las transformaciones y los parámetros utilizados.
- Reproducibilidad: ejecutar el análisis mediante código verificable.
- No realizar *peeking*: no detener o prolongar el experimento solo porque el resultado momentáneo conviene.
- No contaminar grupos: una unidad debe recibir únicamente la variante asignada.
- Distinguir asociación, evidencia experimental y decisión práctica.


# V. Recursos
- Computadora, Google Colab y notebook LAB-D6-AEI-PSAYAN-2026-02.ipynb.
- Librerías numpy, pandas, scipy, statsmodels y plotly.
- Caso ficticio NexaCommerce con datos simulados generados en el propio notebook. No requiere descargar un dataset externo.

## Naturaleza de los datos utilizados
Los registros son sintéticos y reproducibles mediante una semilla fija. Representan usuarios de un checkout; no proceden de una empresa real. Las probabilidades del simulador son supuestos académicos, no evidencia observada. No cambie la semilla para buscar un resultado favorable. En un estudio real, los datos procederían de registros del experimento y el plan se documentaría antes de recogerlos.


# VI. Metodología para el desarrollo de la sesión

El laboratorio se desarrolla mediante una secuencia guiada de cinco momentos:

1. **Comprender:** recuperar el vocabulario esencial.
2. **Representar:** traducir el caso a la estructura de un diseño experimental.
3. **Formular:** establecer las hipótesis y la regla de decisión antes de observar resultados.
4. **Ejecutar e interpretar:** correr el código por etapas y explicar cada salida.
5. **Decidir y transferir:** integrar evidencia estadística, umbral empresarial y guardrails; luego resolver retos breves.

No se solicita exposición, informe ni trabajo adicional. La evidencia de aprendizaje se construye durante la clase mediante respuestas breves en las pausas de análisis.


# VII. Procedimiento

## Actividad 1 — Activación conceptual

**Propósito:** reconocer los componentes que deben definirse antes de ejecutar un experimento.

Complete la tabla con sus propias palabras. La pregunta de apoyo permite verificar si la definición es suficiente.

| Concepto | Definición con sus propias palabras | Pregunta de apoyo |
|---|---|---|
| Experimento A/B | Comparación controlada entre dos versiones (A y B) donde la asignación a cada grupo es aleatoria; a diferencia de solo comparar dos grupos preexistentes, la aleatorización evita que otras variables expliquen la diferencia observada. | ¿Qué diferencia existe entre comparar dos grupos y experimentar? |
| Control | El grupo que mantiene la versión actual (checkout de cuatro pasos), usado como punto de referencia contra el cual se mide cualquier cambio. | ¿Qué representa el punto de referencia? |
| Tratamiento | El grupo que recibe la versión nueva que se quiere evaluar (checkout simplificado de dos pasos). | ¿Qué cambio recibe este grupo? |
| Unidad experimental | El usuario individual, que conserva la misma variante (A o B) durante toda la ventana de observación de siete días. | ¿Sobre quién o qué se aplica la variante? |
| Variable respuesta | Si el usuario compró o no dentro de los siete días posteriores a su asignación (compra_7d), una variable binaria 0/1. | ¿Qué resultado se observa en cada unidad? |
| Métrica primaria | La proporción de usuarios que compraron sobre el total de usuarios asignados a cada grupo (tasa de conversión). | ¿Cuál será el criterio principal de éxito? |
| Guardrail | La tasa de incidencias de pago: una métrica que no es el objetivo del experimento, pero que no debería empeorar más allá de un margen aceptado. | ¿Qué no debería deteriorarse? |
| MDE | El efecto mínimo detectable: la diferencia de conversión más pequeña que el diseño está preparado para detectar con la potencia especificada (aquí, 1,5 puntos porcentuales). | ¿Qué efecto se utiliza para planificar la potencia? |
| Potencia | La probabilidad de detectar el MDE como estadísticamente significativo si ese efecto realmente existe; aquí se fija en 80 %. | ¿Qué capacidad tiene el diseño para detectar el MDE? |
| Peeking | Revisar los resultados parciales del experimento antes de completar la muestra planificada y decidir detenerlo o continuarlo según convenga; esto infla la tasa de falsos positivos. | ¿Por qué no se debe decidir mirando resultados parciales? |

**Pausa de verificación:** explique a un compañero por qué el MDE no debe elegirse después de conocer la diferencia observada.

**Respuesta:** Porque el MDE determina el tamaño de muestra necesario *antes* de recolectar datos. Si se elige después de ver el resultado, se puede ajustar el MDE al valor que sea conveniente para que el experimento "parezca" bien diseñado o para justificar una decisión ya tomada, lo cual rompe las garantías estadísticas del diseño (es una forma de manipulación de resultados, análoga al *peeking*).


## Actividad 2 — Interpretación y representación metodológica
NexaCommerce, empresa ficticia, compara el checkout actual de cuatro pasos con uno simplificado de dos pasos. Cada usuario recibe una sola variante. Se mide compra e incidencia de pago dentro de siete días; incluso quienes no compran permanecen en el denominador.

### 2.1 Interpretación
Complete antes de ejecutar: Determinar si el checkout simplificado de dos pasos (B) aumenta la proporción de usuarios que compran dentro de siete días en una magnitud que supere el umbral empresarial de +1,0 pp, sin deteriorar la tasa de incidencias de pago más allá del margen de +0,5 pp, en comparación con el checkout actual de cuatro pasos (A).

### 2.2 Representación del diseño
| Elemento | Definición en este experimento |
|---|---|
| Factor y niveles | Checkout: A actual; B simplificado |
| Unidad experimental | Usuario único con asignación persistente |
| Respuesta primaria | compra_7d: 1 si compró dentro de siete días; 0 si no |
| Métrica primaria | Usuarios que compraron / todos los usuarios asignados |
| Guardrail | incidencia_pago_7d: al menos una incidencia por usuario en siete días |
| Denominador del guardrail | Todos los usuarios asignados, compren o no |
| Margen de deterioro | B−A debe ser inferior a +0,5 pp en incidencias |
| Baseline histórico supuesto | Conversión 11,8 %; incidencias 1,8 % |
| MDE de conversión | +1,5 pp para planificar potencia de 80 % frente a cero |
| Umbral empresarial | Mejora de conversión superior a +1,0 pp |
| Horizonte | Reclutamiento planificado de 14 días y 7 días de maduración |
| Muestra | Se calcula antes de simular las respuestas |

### 2.3 Hipótesis y reglas previas
Conversión: H0: pB−pA ≤ 0; H1: pB−pA > 0. Alfa = 0,05 unilateral.

Relevancia empresarial: exigir límite inferior unilateral del 95 % mayor que 0,010. Esto evalúa un umbral distinto de cero y no se deduce solo de la estimación puntual.

Guardrail: H0: qB−qA ≥ 0,005; H1: qB−qA < 0,005. Exigir límite superior unilateral del 95 % menor que 0,005. No encontrar una diferencia significativa no demuestra que el guardrail esté protegido.

La recomendación combina ambos requisitos; se trata de una regla conjunta, no de escoger la prueba favorable. Los intervalos individuales no deben describirse como una región de confianza simultánea.

**Comprobación:** ¿por qué se comparan proporciones? ¿Cómo cambiarían las hipótesis si interesara cualquier diferencia, positiva o negativa?

**Respuesta:** Se comparan proporciones porque la variable respuesta es binaria (compra sí/no); la proporción de "unos" en cada grupo es el resumen natural de una variable así, y la diferencia de dos proporciones es el objeto de interés, no una media continua ni una varianza (por eso no aplica ANOVA). Si interesara cualquier diferencia, positiva o negativa, las hipótesis pasarían a ser bilaterales: H0: pB−pA = 0 frente a H1: pB−pA ≠ 0, y el contraste usaría alfa/2 en cada cola (por ejemplo, z de 1,96 en vez de 1,645 para 95 %), lo que exige una diferencia observada más grande para alcanzar significancia con el mismo nivel de confianza.


## Actividad 3 — Preparación y planificación previa
### Paso 1. Preparar Google Colab
Ejecute las celdas en orden. La configuración y el cálculo de muestra aparecen antes de generar las respuestas. En Colab, la primera celda instala solo los paquetes que faltan. Si se solicita reiniciar, vuelva a ejecutar desde aquí.


In [ ]:
import importlib.util
import subprocess
import sys

paquetes = ["numpy", "pandas", "scipy", "statsmodels", "plotly"]
faltan = [p for p in paquetes if importlib.util.find_spec(p) is None]
if faltan:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
print("Dependencias disponibles.")


Dependencias disponibles.


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats import norm
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

pd.set_option("display.max_columns", 20)
print("Entorno listo.")


Entorno listo.


### Paso 2. Fijar los parámetros antes de observar
Los baselines son supuestos históricos del caso ficticio, no estimaciones obtenidas de los grupos que analizaremos. El margen del guardrail se fija por criterio de negocio. Se calcula muestra para conversión y, por aproximación normal, para no inferioridad del guardrail suponiendo diferencia verdadera cero. Se toma el mayor tamaño. Las potencias se refieren a cada objetivo de diseño; no garantizan 80 % para la regla conjunta ni para superar el umbral empresarial.


In [ ]:
ALPHA = 0.05
POTENCIA = 0.80
BASELINE = 0.118
MDE = 0.015
UMBRAL_NEGOCIO = 0.010
BASE_GUARDRAIL = 0.018
MARGEN_GUARDRAIL = 0.005
SEMILLA = 20260921
DIAS_RECLUTAMIENTO = 14
VENTANA_DIAS = 7

assert 0 < BASELINE < BASELINE + MDE < 1
assert 0 < UMBRAL_NEGOCIO < MDE
analysis = NormalIndPower()
es = proportion_effectsize(BASELINE + MDE, BASELINE)
n_conversion = int(np.ceil(analysis.solve_power(
    effect_size=es, alpha=ALPHA, power=POTENCIA,
    ratio=1, alternative="larger")))
z_alpha = norm.ppf(1 - ALPHA)
z_power = norm.ppf(POTENCIA)
n_guardrail = int(np.ceil(
    2 * BASE_GUARDRAIL * (1 - BASE_GUARDRAIL)
    * (z_alpha + z_power)**2 / MARGEN_GUARDRAIL**2))
N_POR_GRUPO = max(n_conversion, n_guardrail)
print("Muestra por grupo para conversión:", n_conversion)
print("Muestra aproximada por grupo para guardrail:", n_guardrail)
print("Muestra planificada por grupo:", N_POR_GRUPO)
print("Analizar una sola vez al completar muestra y seguimiento.")


Muestra por grupo para conversión: 6027
Muestra aproximada por grupo para guardrail: 8743
Muestra planificada por grupo: 8743
Analizar una sola vez al completar muestra y seguimiento.


### Pausa de análisis 1
1. ¿Por qué el baseline debe establecerse antes de ver el resultado?
2. ¿Por qué la muestra final considera también el guardrail?
3. ¿La potencia calculada garantiza demostrar una mejora superior a 1,0 pp?

**Respuestas:**

1. Porque el baseline es un supuesto histórico usado para calcular cuántos usuarios se necesitan; si se fijara después de observar los datos del propio experimento, se podría elegir un valor que haga parecer "exitoso" cualquier resultado, perdiendo la garantía de que el diseño fue planificado de forma independiente del resultado.
2. Porque el guardrail (incidencias de pago) también necesita suficiente muestra para detectar un deterioro relevante con la potencia deseada. Al ejecutar el cálculo, la muestra que exige el guardrail (8 743 por grupo) resultó mayor que la que exige la conversión (6 027 por grupo), así que se toma la mayor de las dos (N_POR_GRUPO = 8 743) para que ambos objetivos queden cubiertos.
3. No. La potencia de 80 % se calculó para detectar el MDE de 1,5 pp frente a **cero**, no para demostrar que la mejora supera el umbral empresarial de 1,0 pp. Son dos preguntas distintas: una es "¿hay algún efecto?" y otra es "¿el efecto es suficientemente grande para el negocio?". El diseño no garantiza potencia de 80 % para esta segunda pregunta, que es más exigente.


### Paso 3. Generar y validar el conjunto académico
El simulador asigna aleatoriamente el mismo número de usuarios a A y B. Genera una fila por usuario y completa siete días de observación para todos. Supone conversiones de 11,8 % y 13,8 % e incidencias de 1,8 % en ambos grupos. Son parámetros del generador, no resultados que deban recuperarse exactamente. Las respuestas se simulan independientes para simplificar el ejercicio; una empresa real puede presentar dependencias y estacionalidad no representadas aquí.


In [ ]:
rng = np.random.default_rng(SEMILLA)
total = 2 * N_POR_GRUPO
grupos = rng.permutation(np.repeat(["A_Control", "B_Tratamiento"], N_POR_GRUPO))
prob_compra = np.where(grupos == "A_Control", 0.118, 0.138)
df = pd.DataFrame({
    "usuario_id": np.arange(1, total + 1),
    "grupo": grupos,
    "dia_asignacion": rng.integers(1, DIAS_RECLUTAMIENTO + 1, total),
    "dias_observados": np.full(total, VENTANA_DIAS),
    "compra_7d": rng.binomial(1, prob_compra),
    "incidencia_pago_7d": rng.binomial(1, BASE_GUARDRAIL, total)
})
print("DATOS SIMULADOS — NexaCommerce")
print("Dimensión:", df.shape)
display(df.head())


DATOS SIMULADOS — NexaCommerce
Dimensión: (17486, 6)


,usuario_id,grupo,dia_asignacion,dias_observados,compra_7d,incidencia_pago_7d
0,1,A_Control,8,7,0,0
1,2,B_Tratamiento,5,7,0,0
2,3,A_Control,12,7,0,0
3,4,A_Control,9,7,0,0
4,5,B_Tratamiento,12,7,0,0


In [ ]:
requeridas = {"usuario_id", "grupo", "dias_observados",
              "compra_7d", "incidencia_pago_7d"}
assert requeridas.issubset(df.columns), "Faltan columnas"
assert not df[list(requeridas)].isna().any().any(), "Hay datos ausentes"
assert df["usuario_id"].is_unique, "Usuarios repetidos o contaminados"
assert set(df["grupo"]) == {"A_Control", "B_Tratamiento"}
assert (df["dias_observados"] == VENTANA_DIAS).all()
for col in ["compra_7d", "incidencia_pago_7d"]:
    assert df[col].isin([0, 1]).all(), f"{col} no es binaria"
ab = df.copy()
tam_grupos = ab.groupby("grupo").size()
assert (tam_grupos == N_POR_GRUPO).all()
display(tam_grupos.to_frame("usuarios"))
print("Calidad estructural y seguimiento completos.")


,usuarios
grupo,
A_Control,8743
B_Tratamiento,8743


Calidad estructural y seguimiento completos.


### Pausa de análisis 2
1. Registre el tamaño de cada grupo. ¿Cumple el plan?
2. ¿Qué ocurriría si un usuario apareciera en ambas variantes?
3. ¿Por qué no debe excluirse a quienes no compraron?
4. ¿Por qué es necesario completar los siete días antes de analizar?

**Respuestas:**

1. Cada grupo tiene 8 743 usuarios (A_Control: 8 743; B_Tratamiento: 8 743), para un total de 17 486. Coincide exactamente con N_POR_GRUPO = 8 743 calculado en el paso anterior, así que el plan se cumple.
2. Se estaría contaminando el experimento: ese usuario recibiría el efecto de ambas variantes a la vez, lo que rompe el supuesto de que cada unidad experimental recibe *una sola* condición. La diferencia observada entre A y B ya no reflejaría el efecto real del checkout, y las varianzas usadas en el contraste (que asumen grupos independientes) dejarían de ser válidas.
3. Porque el denominador de la métrica primaria son *todos* los usuarios asignados, no solo los que compraron. Excluir a quienes no compraron sesgaría la tasa de conversión (dejaría de medir "qué proporción del total compra" para medir otra cosa) y probablemente favorecería artificialmente a un grupo si la tasa de abandono difiere entre A y B.
4. Porque la variable respuesta se define como "compra dentro de los siete días posteriores a la asignación". Si se analizara antes de que todos los usuarios completen esa ventana, los usuarios asignados más recientemente tendrían menos tiempo para convertir que los asignados antes, introduciendo un sesgo de tiempo incompleto que subestimaría la conversión real de ambos grupos (y posiblemente de forma desigual entre A y B).


## Actividad 4 — MDE, potencia y sensibilidad de muestra
### Paso 4. Recuperar el diseño fijado
Revise los parámetros sin recalibrarlos con el resultado observado. El MDE sirve para dimensionar la muestra; el umbral empresarial se utiliza para valorar la mejora. En este caso, las dos cantidades difieren deliberadamente.


In [ ]:
display(pd.DataFrame({
    "parametro": ["Baseline", "MDE", "Umbral empresarial",
                  "Margen guardrail", "Alfa", "Potencia"],
    "valor": [BASELINE, MDE, UMBRAL_NEGOCIO,
              MARGEN_GUARDRAIL, ALPHA, POTENCIA]
}))
print("Muestra final por grupo:", N_POR_GRUPO)


,parametro,valor
0,Baseline,0.118
1,MDE,0.015
2,Umbral empresarial,0.010
3,Margen guardrail,0.005
4,Alfa,0.050
5,Potencia,0.800


Muestra final por grupo: 8743


### Paso 5. Explorar la sensibilidad al MDE
Compare MDE de 0,5, 1,5 y 2,5 pp. Esta exploración es didáctica y no cambia el plan ni la decisión del caso ya fijado.
**Predicción:** ¿qué efecto necesitará más muestra y por qué?


In [ ]:
filas = []
for mde in [0.005, 0.015, 0.025]:
    es = proportion_effectsize(BASELINE + mde, BASELINE)
    n = analysis.solve_power(effect_size=es, alpha=ALPHA,
                             power=POTENCIA, alternative="larger")
    filas.append({"MDE_pp": mde * 100,
                  "n_conversion": int(np.ceil(n)),
                  "n_con_guardrail": max(int(np.ceil(n)), n_guardrail)})
sensibilidad_n = pd.DataFrame(filas)
display(sensibilidad_n)
fig = px.line(sensibilidad_n, x="MDE_pp",
              y=["n_conversion", "n_con_guardrail"], markers=True,
              title="Muestra por grupo según el MDE",
              labels={"value": "Usuarios por grupo", "MDE_pp": "MDE (pp)"})
fig.show()


,MDE_pp,n_conversion,n_con_guardrail
0,0.5,52413,52413
1,1.5,6027,8743
2,2.5,2241,8743


### Pausa de análisis 3
1. Explique por qué reducir el MDE aumenta la muestra para conversión.
2. ¿Por qué el guardrail puede determinar la muestra final?
3. ¿Sería válido aumentar el MDE después de ver los resultados para declarar éxito?

**Respuestas:**

1. Un MDE más pequeño significa que se quiere detectar una diferencia más sutil entre A y B. Como esa diferencia es más difícil de distinguir del ruido muestral, se necesitan más observaciones para poder separarla estadísticamente del cero con la misma confianza y potencia. Esto se ve directamente en la tabla: bajar el MDE de 1,5 pp a 0,5 pp multiplica la muestra de 6 027 a 52 413 por grupo — casi 9 veces más.
2. Porque el guardrail tiene su propio requisito de muestra (calculado en el paso 2), y el diseño final toma el **máximo** entre ambos requisitos para asegurar que los dos objetivos —detectar el MDE de conversión y poder evaluar el guardrail con suficiente precisión— queden cubiertos con una sola muestra. En este caso, para MDE de 1,5 pp y 2,5 pp la muestra del guardrail (8 743) es mayor que la de conversión, así que es el guardrail el que termina fijando el tamaño final.
3. No. Aumentar el MDE después de observar los resultados sería ajustar la regla de decisión a conveniencia para que el resultado ya obtenido "califique" como éxito. Esto invalida las garantías estadísticas del diseño (nivel de significancia y potencia reales) porque la regla dejó de fijarse antes de ver los datos; es una forma de sesgo de selección de resultados, igual de problemática que el *peeking*.


## Actividad 5 — Análisis estadístico de la comparación A/B

**Propósito:** calcular el efecto observado y evaluar la hipótesis mediante una prueba para dos proporciones.

### Paso 6. Obtener tasas y tamaño del efecto observado

Antes de ejecutar, recuerde:

- Diferencia absoluta: $p_B-p_A$.
- Lift relativo: $(p_B-p_A)/p_A$.
- La diferencia se expresa en puntos porcentuales; el lift expresa un cambio relativo respecto de A.


In [ ]:
tabla = ab.groupby("grupo")["compra_7d"].agg(["sum", "count", "mean"])
tabla.columns = ["compras", "n", "tasa_conversion"]
x_A, n_A = int(tabla.loc["A_Control", "compras"]), int(tabla.loc["A_Control", "n"])
x_B, n_B = int(tabla.loc["B_Tratamiento", "compras"]), int(tabla.loc["B_Tratamiento", "n"])
p_A, p_B = x_A / n_A, x_B / n_B
efecto = p_B - p_A
lift_rel = efecto / p_A if p_A > 0 else np.nan
display(tabla)
print(f"Diferencia B−A: {efecto*100:.3f} pp")
print(f"Lift relativo: {lift_rel:.2%}")


,compras,n,tasa_conversion
grupo,,,
A_Control,996,8743,0.11392
B_Tratamiento,1236,8743,0.14137


Diferencia B−A: 2.745 pp
Lift relativo: 24.10%


### Pausa de análisis 4

Complete sin utilizar todavía el p-valor:

> La tasa de B es 14,14 % y la tasa de A es 11,39 %. La diferencia observada es 2,75 puntos porcentuales, equivalente a un lift relativo de 24,10 %. Estos valores describen la muestra, pero todavía no permiten afirmar que esa diferencia sea estadísticamente significativa ni que refleje un efecto real de la variante B y no solo variación aleatoria propia de la muestra observada.


### Paso 7. Contraste e intervalos coherentes
Usamos la aproximación normal de Wald con error estándar no combinado. Se comprueba que haya al menos diez compras y diez no compras en cada grupo; esta regla práctica no sustituye la revisión del diseño.

Para la prueba unilateral frente a cero, p < 0,05 equivale a límite inferior unilateral del 95 % > 0 con el mismo método. El IC bilateral del 95 % se presenta aparte. La función también permite contrastar un umbral distinto de cero. El p-valor no es la probabilidad de que H0 sea verdadera.


In [ ]:
def diferencia_wald(x_b, n_b, x_a, n_a, umbral=0.0):
    if min(x_b, n_b-x_b, x_a, n_a-x_a) < 10:
        raise ValueError("Frecuencias pequeñas: revisar método antes de continuar")
    pb, pa = x_b/n_b, x_a/n_a
    d = pb-pa
    se = np.sqrt(pb*(1-pb)/n_b + pa*(1-pa)/n_a)
    z = (d-umbral)/se
    return {"d": d, "se": se, "z": z,
            "p_mayor": norm.sf(z), "p_menor": norm.cdf(z),
            "li_uni95": d-norm.ppf(1-ALPHA)*se,
            "ls_uni95": d+norm.ppf(1-ALPHA)*se,
            "ic95_low": d-norm.ppf(1-ALPHA/2)*se,
            "ic95_high": d+norm.ppf(1-ALPHA/2)*se}

conv = diferencia_wald(x_B, n_B, x_A, n_A)
negocio = diferencia_wald(x_B, n_B, x_A, n_A, UMBRAL_NEGOCIO)
print(f"z frente a cero: {conv['z']:.4f}")
print(f"p unilateral frente a cero: {conv['p_mayor']:.6g}")
print(f"Límite inferior unilateral 95%: {conv['li_uni95']*100:.3f} pp")
print(f"IC bilateral 95%: [{conv['ic95_low']*100:.3f}, {conv['ic95_high']*100:.3f}] pp")
print(f"p unilateral frente a +1,0 pp: {negocio['p_mayor']:.6g}")


z frente a cero: 5.4436
p unilateral frente a cero: 2.61091e-08
Límite inferior unilateral 95%: 1.916 pp
IC bilateral 95%: [1.757, 3.733] pp
p unilateral frente a +1,0 pp: 0.000269553


In [ ]:
grafico = tabla.reset_index()
grafico["conversion_pct"] = grafico["tasa_conversion"] * 100
fig = px.bar(grafico, x="grupo", y="conversion_pct", text="conversion_pct",
             title="Conversión observada — datos simulados",
             labels={"conversion_pct": "Usuarios que compraron (%)", "grupo": "Grupo"})
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_yaxes(rangemode="tozero")
fig.show()


### Análisis y conclusión estadística
1. Registre alfa, z y p unilateral frente a cero.
2. ¿Se rechaza H0? Explique en el contexto de conversión del caso simulado.
3. ¿El límite inferior unilateral supera cero? ¿Y el umbral de 1,0 pp?
4. Explique por qué el IC bilateral del 95 % y el límite unilateral del 95 % tienen extremos distintos.
5. ¿Superar 1,0 pp con la estimación puntual demuestra superar ese umbral con evidencia suficiente?

**Respuestas:**

1. Alfa = 0,05; z = 5,44; p unilateral frente a cero ≈ 2,6 × 10⁻⁸.
2. Sí, se rechaza H0. Un p-valor de 2,6 × 10⁻⁸ es muchísimo menor que 0,05, por lo que hay evidencia estadística muy fuerte de que la conversión de B es mayor que la de A en este experimento simulado (no se puede atribuir la diferencia observada al azar).
3. Sí a ambas. El límite inferior unilateral del 95 % es 1,92 pp, que supera tanto el cero (confirma el rechazo de H0) como el umbral empresarial de 1,0 pp (confirma relevancia práctica, no solo significancia estadística).
4. Porque usan puntos críticos distintos de la misma distribución normal: el límite unilateral del 95 % usa z = 1,645 (deja 5 % en una sola cola), mientras que el IC bilateral del 95 % usa z = 1,96 (deja 2,5 % en cada cola) para poder acotar el efecto por ambos lados a la vez. Por eso el límite inferior unilateral (1,92 pp) es mayor que el límite inferior del IC bilateral (1,76 pp): el unilateral "gasta" todo su margen de error de un solo lado.
5. No. La estimación puntual (2,75 pp) superar el umbral no es, por sí sola, evidencia suficiente, porque una estimación puntual no dice nada sobre la incertidumbre muestral. Lo que sí constituye evidencia suficiente es que el **límite inferior** del intervalo unilateral (1,92 pp) también supere el umbral (1,0 pp): eso significa que, incluso en el escenario menos favorable compatible con los datos al 95 % de confianza, la mejora seguiría superando el umbral empresarial.


## Actividad 6 — Del resultado estadístico a la decisión
### Paso 8. Evaluar el guardrail con incertidumbre
El guardrail es la proporción de usuarios con alguna incidencia de pago en siete días. El margen de deterioro admisible es +0,5 pp. La no inferioridad exige que el límite superior unilateral del 95 % esté por debajo de ese margen. No basta obtener p > 0,05 en una prueba de igualdad.


In [ ]:
guardrail = ab.groupby("grupo")["incidencia_pago_7d"].agg(["sum", "count", "mean"])
guardrail.columns = ["incidencias", "n", "tasa_incidencia"]
g_A = int(guardrail.loc["A_Control", "incidencias"])
g_B = int(guardrail.loc["B_Tratamiento", "incidencias"])
guard = diferencia_wald(g_B, n_B, g_A, n_A, MARGEN_GUARDRAIL)
display(guardrail)
print(f"Cambio del guardrail B−A: {guard['d']*100:.3f} pp")
print(f"Límite superior unilateral 95%: {guard['ls_uni95']*100:.3f} pp")
print(f"Margen permitido: {MARGEN_GUARDRAIL*100:.1f} pp")
print(f"p de no inferioridad: {guard['p_menor']:.6g}")


,incidencias,n,tasa_incidencia
grupo,,,
A_Control,139,8743,0.015898
B_Tratamiento,176,8743,0.020130


Cambio del guardrail B−A: 0.423 pp
Límite superior unilateral 95%: 0.754 pp
Margen permitido: 0.5 pp
p de no inferioridad: 0.351286


### Paso 9. Integrar la regla de decisión
| Criterio | Evidencia requerida |
|---|---|
| Diseño y calidad | Una fila por usuario, asignación persistente, muestra y seguimiento completos |
| Mejora frente a cero | p unilateral < 0,05 |
| Relevancia empresarial | Límite inferior unilateral del 95 % > +1,0 pp |
| Guardrail | Límite superior unilateral del 95 % < +0,5 pp |

La regla se fijó antes de simular. El MDE dimensiona la muestra y no reemplaza al umbral empresarial. Si no se satisface un requisito, distinguir evidencia insuficiente de evidencia de perjuicio. Cualquier experimento posterior necesita un nuevo plan; no seguir agregando datos hasta lograr significancia.


In [ ]:
cumple_estadistica = bool(conv["p_mayor"] < ALPHA)
cumple_negocio = bool(conv["li_uni95"] > UMBRAL_NEGOCIO)
cumple_guardrail = bool(guard["ls_uni95"] < MARGEN_GUARDRAIL)
cumple_diseno = bool((tam_grupos >= N_POR_GRUPO).all())
criterios = pd.Series({"Diseño validado": cumple_diseno,
    "Mejora frente a cero": cumple_estadistica,
    "Mejora superior al umbral": cumple_negocio,
    "Guardrail protegido": cumple_guardrail})
display(criterios.to_frame("cumple"))
if criterios.all():
    decision = "Recomendar B en el caso simulado, con seguimiento posterior."
else:
    decision = "No recomendar el despliegue todavía: revisar los criterios no cumplidos."
print(decision)
print("Conclusión académica con datos simulados; no valida un despliegue real.")


,cumple
Diseño validado,True
Mejora frente a cero,True
Mejora superior al umbral,True
Guardrail protegido,False


No recomendar el despliegue todavía: revisar los criterios no cumplidos.
Conclusión académica con datos simulados; no valida un despliegue real.


### Pausa de decisión
| Componente | Resultado observado | Interpretación |
|---|---|---|
| p unilateral frente a cero | 2,6 × 10⁻⁸ | Muy inferior a 0,05: se rechaza H0, la mejora de conversión no es atribuible al azar. |
| Diferencia B−A en pp y lift relativo | 2,75 pp; lift de 24,10 % | B convierte notablemente más que A en este experimento simulado. |
| Límite inferior y umbral empresarial | 1,92 pp > 1,0 pp | La mejora supera el umbral empresarial incluso en el escenario más conservador del intervalo. |
| Límite superior del guardrail y margen | 0,75 pp > 0,5 pp | El límite supera el margen permitido: no hay evidencia de que el guardrail se mantenga dentro de lo aceptable. |
| Calidad y límites del caso simulado | Diseño completo (8 743 por grupo, sin datos ausentes ni contaminación) | La calidad estructural del experimento es adecuada; los límites son los propios de datos simulados e independientes, no de una empresa real. |

**Decisión en cuatro líneas:** No recomendar el despliegue de B todavía, porque aunque la mejora de conversión es clara y supera el umbral empresarial (límite inferior de 1,92 pp frente a un umbral de 1,0 pp, con p ≈ 2,6 × 10⁻⁸), el guardrail de incidencias de pago no queda protegido: su límite superior unilateral (0,75 pp) excede el margen permitido (0,5 pp). Esto no demuestra que B cause más incidencias de las aceptables, solo que la evidencia actual no permite descartarlo. Se propone un nuevo experimento enfocado en reducir la incertidumbre del guardrail (mayor muestra o mayor ventana de observación) antes de decidir el despliegue.


## Actividad 7 — Retos de aplicación y retroalimentación
Se resuelven durante la clase y no generan una entrega adicional.

### Reto 1 — Estimación y umbral
Un caso presenta p = 0,02 frente a cero, mejora estimada de 0,4 pp y umbral empresarial de 3 pp. ¿Qué demuestra la prueba y qué no demuestra? ¿Qué información aporta el intervalo?

**Respuesta:** La prueba demuestra que la diferencia es estadísticamente distinta de cero (p = 0,02 < 0,05): hay un efecto real, no es solo ruido. Pero no demuestra que ese efecto sea relevante para el negocio, porque la mejora estimada (0,4 pp) está muy por debajo del umbral empresarial (3 pp). El intervalo de confianza es lo que permite ver esto con claridad: si su límite superior está lejos de 3 pp, aporta evidencia de que el efecto, aunque real, es demasiado pequeño para justificar el despliegue por motivos de negocio.

### Reto 2 — Cambio de pregunta
Si interesa cualquier diferencia entre A y B, formule H0 y H1 e indique el tipo de contraste. ¿Por qué debe elegirse antes de observar los resultados?

**Respuesta:** H0: pB−pA = 0; H1: pB−pA ≠ 0. El contraste es bilateral (de dos colas), porque ahora interesa detectar una diferencia en cualquier dirección, no solo que B sea mejor que A. Debe elegirse antes de observar los resultados porque, de lo contrario, se podría escoger el tipo de contraste (unilateral o bilateral) según cuál favorezca la conclusión deseada después de ver los datos, lo cual es una forma de manipulación de resultados (p-hacking) que invalida las garantías de error del análisis.

### Reto 3 — Guardrail no concluyente
El margen es +0,5 pp. El aumento estimado de incidencias es +0,2 pp, pero el límite superior unilateral es +0,7 pp. ¿Se ha demostrado cumplir el guardrail? ¿Es correcto afirmar que se ha demostrado perjuicio?

**Respuesta:** No se ha demostrado cumplir el guardrail: aunque la estimación puntual (0,2 pp) está dentro del margen, el límite superior del intervalo (0,7 pp) lo supera, así que no se puede descartar que el verdadero deterioro sea mayor al permitido. Tampoco es correcto afirmar que se ha demostrado perjuicio: el intervalo solo indica que la evidencia es insuficiente para garantizar que el guardrail está protegido, no que exista prueba de daño. Es un resultado no concluyente, y debe tratarse como tal, no como un "sí" ni como un "no" definitivo.

### Reto 4 — Reproducibilidad y sesgo
Un equipo cambia la semilla hasta conseguir una decisión favorable. Explique por qué eso invalida el ejercicio. Proponga cómo documentar una exploración de escenarios sin sustituir el caso base.

**Respuesta:** Cambiar la semilla repetidamente hasta obtener el resultado deseado es una forma de p-hacking: al probar muchas corridas y quedarse solo con la favorable, la probabilidad real de obtener "significancia" por puro azar deja de ser el 5 % nominal y puede ser mucho mayor, invalidando las garantías del contraste de hipótesis. Para explorar escenarios sin caer en esto, se debe declarar de antemano cuántas semillas se van a probar y con qué propósito (por ejemplo, evaluar la estabilidad del resultado), reportar los resultados de *todas* las corridas —no solo la conveniente— y mantener el caso base (la semilla y el diseño originalmente pre-registrados) como la conclusión oficial del experimento.


# VIII. Secuencia de trabajo para la sesión
| Momento | Minutos | Desarrollo |
|---|---:|---|
| Apertura | 10 | Pregunta orientadora y caso |
| Actividad 1 | 15 | Activación conceptual |
| Actividad 2 | 20 | Representación e hipótesis |
| Actividad 3 | 30 | Planificación, simulación y calidad |
| Receso | 15 | Pausa |
| Actividad 4 | 25 | Sensibilidad de muestra |
| Actividad 5 | 30 | Tasas, contraste e intervalos |
| Actividad 6 | 20 | Guardrail y decisión |
| Actividad 7 | 15 | Retos y ticket de salida |

Total: 180 minutos, incluido el receso.

## Ticket de salida
«Un experimento A/B no termina en el p-valor porque…»

**Respuesta:** …un p-valor pequeño solo dice que el efecto observado probablemente no es azar, pero no dice si ese efecto es lo bastante grande para el negocio, ni si otras métricas importantes (guardrails) siguen protegidas. La decisión final exige integrar tres piezas de evidencia —significancia estadística, relevancia práctica y guardrails— y solo cuando las tres son favorables se justifica avanzar con el despliegue.


# IX. Lista de verificación de aprendizaje
- [ ] Identifico factor, niveles y unidad experimental.
- [ ] Defino respuesta, denominador y ventana de observación.
- [ ] Formulo hipótesis antes de observar resultados.
- [ ] Distingo baseline, MDE, umbral empresarial, alfa y potencia.
- [ ] Calculo muestra antes de generar los datos del caso.
- [ ] Interpreto pp, lift, p-valor e intervalos sin confundirlos.
- [ ] Evalúo el guardrail con margen e incertidumbre.
- [ ] Distingo evidencia insuficiente de evidencia de perjuicio.
- [ ] Reconozco peeking, contaminación y selección de resultados.
- [ ] Declaro el carácter simulado y los límites del ejercicio.


# X. Referencias
- Kohavi, R., Tang, D., y Xu, Y. (2020). Trustworthy Online Controlled Experiments. Cambridge University Press.
- Bruce, P., Bruce, A., y Gedeck, P. (2020). Practical Statistics for Data Scientists, segunda edición. O’Reilly.
- statsmodels. Documentación de NormalIndPower y proportion_effectsize: https://www.statsmodels.org/stable/stats.html
- SciPy. Distribución normal, funciones cdf, sf y ppf: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html

Datos: simulación académica NexaCommerce incluida en el notebook, semilla 20260921. Los datos no proceden de las referencias bibliográficas.
